In [1]:
from pathlib import Path
import sys

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "requirements.txt").is_file()
            and (candidate / "libs").is_dir()
        ):
            return candidate

    raise RuntimeError("Project root could not be found.")

ROOT = find_project_root()

DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"

REPORTS_DIR = RESULTS_DIR / "reports"
CURVES_DIR = RESULTS_DIR / "curves_data"
AGGREGATED_REPORTS_DIR = RESULTS_DIR / "aggregated_reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CURVES_DIR.mkdir(parents=True, exist_ok=True)
AGGREGATED_REPORTS_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split

# ==========================================
# 1. SETTINGS
# ==========================================
# Replace with the path and name of your original file
CAMINHO_DATASET_ORIGINAL = DATA_DIR / "cicids2017" / "cicids2017_cleaned.csv" 
PASTA_SAIDA = DATA_DIR / "cicids2017"

# ==========================================
# 2. DATA LOADING
# ==========================================
print(f"Reading the original dataset: {CAMINHO_DATASET_ORIGINAL}...")
df = pd.read_csv(CAMINHO_DATASET_ORIGINAL)

print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")

# Remove whitespace from column names
df.columns = df.columns.str.strip()

# ==========================================
# 3. TARGET COLUMN TRANSFORMATION
# ==========================================
print("Generating the binary 'label' column and formatting 'attack_cat'...")

# Convert everything to lowercase and strip spaces to avoid formatting-related failures
df['Attack Type'] = df['Attack Type'].astype(str).str.strip().str.lower()

# The original string is 'Normal Traffic'. In lowercase it becomes 'normal traffic'
TERMO_NORMAL = 'normal traffic'

# Create the 'label' column (Binary: 0 for Normal, 1 for Attack)
df['label'] = df['Attack Type'].apply(lambda x: 0 if x == TERMO_NORMAL else 1)

# Rename 'Attack Type' to 'attack_cat' (the column expected by the main code)
df.rename(columns={'Attack Type': 'attack_cat'}, inplace=True)

# Print a short summary to confirm that both classes exist!
contagem_binaria = df['label'].value_counts()
print("\n--- Binary Class Distribution ---")
print(f"Class 0 (Normal): {contagem_binaria.get(0, 0)} packets")
print(f"Class 1 (Attack): {contagem_binaria.get(1, 0)} packets")

# Clean NaN and infinite values
df.fillna(0, inplace=True)
df.replace([np.inf, -np.inf], 0, inplace=True)

# ==========================================
# 4. TRAIN/TEST SPLIT (80% / 20%)
# ==========================================
print("\nSplitting the data into Training and Test sets (Stratified)...")
df_train, df_test = train_test_split(df, test_size=0.20, random_state=42, stratify=df['attack_cat'])

# ==========================================
# 5. SAVE THE FILES
# ==========================================
os.makedirs(PASTA_SAIDA, exist_ok=True)

caminho_treino = os.path.join(PASTA_SAIDA, "CICIDS_training-set.csv")
caminho_teste = os.path.join(PASTA_SAIDA, "CICIDS_testing-set.csv")

print(f"Saving Training set to {caminho_treino}...")
df_train.to_csv(caminho_treino, index=False)

print(f"Saving Test set to {caminho_teste}...")
df_test.to_csv(caminho_teste, index=False)

print("\n🚀 Transformation completed! You can run Standard WiSARD now.")

Reading the original dataset: C:\Users\Lucas\Desktop\Trabalho Mestrado\data\cicids2017\cicids2017_cleaned.csv...
Dataset loaded with 2520751 rows and 53 columns.
Generating the binary 'label' column and formatting 'attack_cat'...

--- Binary Class Distribution ---
Class 0 (Normal): 2095057 packets
Class 1 (Attack): 425694 packets

Splitting the data into Training and Test sets (Stratified)...
Saving Training set to C:\Users\Lucas\Desktop\Trabalho Mestrado\data\cicids2017\CICIDS_training-set.csv...
Saving Test set to C:\Users\Lucas\Desktop\Trabalho Mestrado\data\cicids2017\CICIDS_testing-set.csv...

🚀 Transformation completed! You can run Standard WiSARD now.
